# 🔍 Customer Churn Prediction System
### N.V. Mani Deepika — Data Science Portfolio Project

---

| Detail | Info |
|---|---|
| **Dataset** | Telco Customer Churn (7,043 rows, 21 features) |
| **Models** | Logistic Regression · Random Forest · SVM · XGBoost |
| **Techniques** | EDA · Feature Engineering · SHAP · Threshold Tuning |
| **Output** | Cohort Retention Dashboard |

---


## ⚙️ Step 0 — Install Dependencies
Run this once if you haven't installed these packages yet.

In [ ]:
# Run this cell once to install required packages
import subprocess, sys
packages = ['pandas','numpy','matplotlib','seaborn','scikit-learn','xgboost','shap']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("✅ All packages ready!")

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, f1_score, precision_score, recall_score, accuracy_score
)
from xgboost import XGBClassifier
import shap
import os

# ── Plot style ──
%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#f8fafb',
    'axes.facecolor':   '#f8fafb',
    'axes.edgecolor':   '#dddddd',
    'grid.color':       '#eeeeee',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
})

PALETTE = ['#1a6b4a', '#f472b6', '#fbbf24', '#60a5fa']
print("✅ Libraries imported!")

## 📂 Step 2 — Load Dataset
> **If you have `telco_churn.csv`** already in the same folder, skip the first code cell and run only the second one.  
> **If not**, run the first cell to generate a realistic Telco-style dataset (7,043 rows).


In [ ]:
# ── OPTION A: Generate dataset (run if you don't have the CSV) ──
np.random.seed(42)
n = 7043

gender           = np.random.choice(['Male','Female'], n)
senior           = np.random.choice([0,1], n, p=[0.84,0.16])
partner          = np.random.choice(['Yes','No'], n)
dependents       = np.random.choice(['Yes','No'], n, p=[0.3,0.7])
tenure           = np.random.randint(0, 72, n)
phone_service    = np.random.choice(['Yes','No'], n, p=[0.9,0.1])
multiple_lines   = np.where(phone_service=='No','No phone service', np.random.choice(['Yes','No'],n))
internet_service = np.random.choice(['DSL','Fiber optic','No'], n, p=[0.34,0.44,0.22])
online_security  = np.where(internet_service=='No','No internet service', np.random.choice(['Yes','No'],n))
online_backup    = np.where(internet_service=='No','No internet service', np.random.choice(['Yes','No'],n))
device_protection= np.where(internet_service=='No','No internet service', np.random.choice(['Yes','No'],n))
tech_support     = np.where(internet_service=='No','No internet service', np.random.choice(['Yes','No'],n))
streaming_tv     = np.where(internet_service=='No','No internet service', np.random.choice(['Yes','No'],n))
streaming_movies = np.where(internet_service=='No','No internet service', np.random.choice(['Yes','No'],n))
contract         = np.random.choice(['Month-to-month','One year','Two year'], n, p=[0.55,0.24,0.21])
paperless        = np.random.choice(['Yes','No'], n, p=[0.59,0.41])
payment          = np.random.choice(['Electronic check','Mailed check','Bank transfer (automatic)','Credit card (automatic)'], n)
monthly_charges  = np.round(np.random.uniform(18, 119, n), 2)
total_charges    = np.round(monthly_charges * tenure + np.random.uniform(0,50,n), 2)

churn_prob = (
    0.05
    + 0.25 * (contract == 'Month-to-month')
    + 0.10 * (internet_service == 'Fiber optic')
    + 0.08 * (payment == 'Electronic check')
    + 0.06 * (senior == 1)
    - 0.15 * (tenure > 36)
    - 0.05 * (online_security == 'Yes')
    - 0.04 * (tech_support == 'Yes')
    + np.random.normal(0, 0.05, n)
)
churn_prob = np.clip(churn_prob, 0.02, 0.95)
churn = np.where(np.random.uniform(0,1,n) < churn_prob, 'Yes', 'No')

df = pd.DataFrame({
    'customerID': ['CID-'+str(i).zfill(5) for i in range(n)],
    'gender': gender, 'SeniorCitizen': senior, 'Partner': partner,
    'Dependents': dependents, 'tenure': tenure, 'PhoneService': phone_service,
    'MultipleLines': multiple_lines, 'InternetService': internet_service,
    'OnlineSecurity': online_security, 'OnlineBackup': online_backup,
    'DeviceProtection': device_protection, 'TechSupport': tech_support,
    'StreamingTV': streaming_tv, 'StreamingMovies': streaming_movies,
    'Contract': contract, 'PaperlessBilling': paperless,
    'PaymentMethod': payment, 'MonthlyCharges': monthly_charges,
    'TotalCharges': total_charges, 'Churn': churn
})
df.to_csv('telco_churn.csv', index=False)
print(f"✅ Dataset generated & saved! Shape: {df.shape}")
print(f"   Churn rate: {df['Churn'].eq('Yes').mean()*100:.1f}%")

In [ ]:
# ── OPTION B: Load from CSV (run this always) ──
df = pd.read_csv('telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

print(f"✅ Dataset loaded!")
print(f"   Shape          : {df.shape}")
print(f"   Churn rate     : {df['Churn'].eq('Yes').mean()*100:.1f}%")
print(f"   Missing values : {df.isnull().sum().sum()}")
df.head()

## 📊 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Basic stats ──
print("=== Dataset Info ===")
print(df.dtypes.to_string())
print(f"\nChurn value counts:")
print(df['Churn'].value_counts())

In [ ]:
# ── EDA Overview: 6-panel chart ──
fig = plt.figure(figsize=(18, 11), facecolor='#f8fafb')
fig.suptitle('Exploratory Data Analysis — Customer Churn', fontsize=16, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1. Churn pie
churn_counts = df['Churn'].value_counts()
ax1 = fig.add_subplot(gs[0, 0])
wedges, texts, autotexts = ax1.pie(
    churn_counts, labels=['Retained','Churned'],
    colors=[PALETTE[0], PALETTE[1]],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2))
for at in autotexts: at.set_fontsize(11); at.set_fontweight('bold')
ax1.set_title('Churn Distribution', fontweight='bold')

# 2. Tenure histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df[df['Churn']=='No']['tenure'],  bins=30, alpha=0.7, color=PALETTE[0], label='Retained', edgecolor='white')
ax2.hist(df[df['Churn']=='Yes']['tenure'], bins=30, alpha=0.7, color=PALETTE[1], label='Churned',  edgecolor='white')
ax2.set_xlabel('Tenure (months)'); ax2.set_ylabel('Count')
ax2.set_title('Tenure by Churn', fontweight='bold')
ax2.legend(framealpha=0)

# 3. Contract churn rate
ax3 = fig.add_subplot(gs[0, 2])
cc = df.groupby('Contract')['Churn'].apply(lambda x: (x=='Yes').mean()*100).reset_index()
cc.columns = ['Contract','ChurnRate']
bars = ax3.bar(cc['Contract'], cc['ChurnRate'], color=PALETTE[:3], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, cc['ChurnRate']):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax3.set_title('Churn Rate by Contract', fontweight='bold'); ax3.set_ylim(0, cc['ChurnRate'].max()*1.3)

# 4. Internet service
ax4 = fig.add_subplot(gs[1, 0])
ic = df.groupby('InternetService')['Churn'].apply(lambda x: (x=='Yes').mean()*100).reset_index()
ic.columns = ['InternetService','ChurnRate']
ax4.barh(ic['InternetService'], ic['ChurnRate'], color=PALETTE[:3], edgecolor='white')
for bar, val in zip(ax4.patches, ic['ChurnRate']):
    ax4.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
ax4.set_xlabel('Churn Rate (%)'); ax4.set_title('Churn by Internet Service', fontweight='bold')

# 5. Monthly charges boxplot
ax5 = fig.add_subplot(gs[1, 1])
ax5.boxplot(
    [df[df['Churn']=='No']['MonthlyCharges'], df[df['Churn']=='Yes']['MonthlyCharges']],
    labels=['Retained','Churned'], patch_artist=True,
    boxprops=dict(facecolor='#e8f5ef'),
    medianprops=dict(color=PALETTE[0], linewidth=2.5))
ax5.set_ylabel('Monthly Charges ($)'); ax5.set_title('Monthly Charges vs Churn', fontweight='bold')

# 6. Payment method
ax6 = fig.add_subplot(gs[1, 2])
pc = df.groupby('PaymentMethod')['Churn'].apply(lambda x: (x=='Yes').mean()*100).reset_index()
pc.columns = ['PaymentMethod','ChurnRate']
pc['PaymentMethod'] = pc['PaymentMethod'].str.replace(' (automatic)','',regex=False)
ax6.barh(pc['PaymentMethod'], pc['ChurnRate'], color=PALETTE, edgecolor='white')
for bar, val in zip(ax6.patches, pc['ChurnRate']):
    ax6.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
ax6.set_xlabel('Churn Rate (%)'); ax6.set_title('Churn by Payment Method', fontweight='bold')

plt.savefig('01_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA overview saved as 01_eda_overview.png")

In [ ]:
# ── Correlation Heatmap ──
numeric_cols = ['tenure','MonthlyCharges','TotalCharges','SeniorCitizen']
corr_df = df[numeric_cols].copy()
corr_df['Churn'] = (df['Churn']=='Yes').astype(int)

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_df.corr(), dtype=bool))
sns.heatmap(corr_df.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax, linewidths=0.5,
            cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('02_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Correlation heatmap saved")

## 🔧 Step 4 — Data Preprocessing

In [ ]:
df_model = df.drop(columns=['customerID']).copy()

# Binary encoding
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling','Churn']
le = LabelEncoder()
for col in binary_cols:
    df_model[col] = le.fit_transform(df_model[col])

# One-hot encoding
multi_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
              'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
              'Contract','PaymentMethod']
df_model = pd.get_dummies(df_model, columns=multi_cols, drop_first=True)

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"✅ Preprocessing done!")
print(f"   Train : {X_train.shape[0]:,} rows | Test : {X_test.shape[0]:,} rows")
print(f"   Features after encoding : {X_train.shape[1]}")
print(f"   Class balance (train)   : {dict(y_train.value_counts())}")

## 🤖 Step 5 — Train 4 Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=5,
                                         use_label_encoder=False, eval_metric='logloss',
                                         scale_pos_weight=4, random_state=42, verbosity=0),
}

results = {}
print(f"{'Model':<24} {'AUC':>6} {'F1':>6} {'Recall':>8} {'Precision':>10}")
print("-" * 58)

for name, model in models.items():
    X_tr = X_train_sc if name in ['Logistic Regression','SVM'] else X_train
    X_te = X_test_sc  if name in ['Logistic Regression','SVM'] else X_test

    model.fit(X_tr, y_train)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba),
    }
    r = results[name]
    print(f"{name:<24} {r['roc_auc']:>6.3f} {r['f1']:>6.3f} {r['recall']:>8.3f} {r['precision']:>10.3f}")

print("\n✅ All 4 models trained!")

## 📈 Step 6 — Model Evaluation & Comparison

In [ ]:
# ── Model Comparison Bar Chart ──
metrics = ['accuracy','precision','recall','f1','roc_auc']
metric_labels = ['Accuracy','Precision','Recall','F1 Score','ROC-AUC']

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(metrics))
width = 0.2

for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    bars = ax.bar(x + i*width, vals, width, label=name, color=PALETTE[i], alpha=0.88, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Model Comparison — All 4 Classifiers', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', framealpha=0)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig('03_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ROC Curves ──
fig, ax = plt.subplots(figsize=(8, 6))
for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, color=PALETTE[i], linewidth=2.2,
            label=f"{name} (AUC = {res['roc_auc']:.3f})")
ax.plot([0,1],[0,1],'--', color='#cccccc', linewidth=1.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', framealpha=0.95)
plt.tight_layout()
plt.savefig('04_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion Matrices ──
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
fig.suptitle('Confusion Matrices — All Models', fontweight='bold', fontsize=14)
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Greens',
                linewidths=0.5, linecolor='white',
                xticklabels=['Retained','Churned'],
                yticklabels=['Retained','Churned'], cbar=False)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('05_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Step 7 — Threshold Tuning (XGBoost)
> Lowering the threshold reduces **False Negatives** — customers who churn but we miss.

In [ ]:
xgb_proba = results['XGBoost']['y_proba']
thresholds = np.arange(0.20, 0.80, 0.01)
fn_list, f1_list, recall_list, precision_list = [], [], [], []

fn_default = confusion_matrix(y_test, (xgb_proba >= 0.5).astype(int))[1][0]
f1_default = f1_score(y_test, (xgb_proba >= 0.5).astype(int))

for t in thresholds:
    y_pred_t = (xgb_proba >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    fn_list.append(cm[1][0])
    f1_list.append(f1_score(y_test, y_pred_t, zero_division=0))
    recall_list.append(recall_score(y_test, y_pred_t, zero_division=0))
    precision_list.append(precision_score(y_test, y_pred_t, zero_division=0))

best_idx       = np.argmin(fn_list)
best_threshold = thresholds[best_idx]
fn_tuned       = fn_list[best_idx]
fn_reduction   = ((fn_default - fn_tuned) / fn_default) * 100

print(f"Default threshold : 0.50  →  FN = {fn_default}  |  F1 = {f1_default:.3f}")
print(f"Tuned  threshold  : {best_threshold:.2f}  →  FN = {fn_tuned}  |  F1 = {f1_list[best_idx]:.3f}")
print(f"\n🎯 False Negative Reduction : {fn_reduction:.1f}%  ({fn_default} → {fn_tuned})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Tuning — XGBoost', fontweight='bold', fontsize=14)

ax = axes[0]
ax.plot(thresholds, fn_list, color=PALETTE[1], linewidth=2.5, label='False Negatives')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5, label=f'Default 0.50  (FN={fn_default})')
ax.axvline(best_threshold, color=PALETTE[0], linestyle='-', linewidth=2, label=f'Tuned {best_threshold:.2f}  (FN={fn_tuned})')
ax.fill_between(thresholds, fn_list, alpha=0.12, color=PALETTE[1])
ax.set_xlabel('Threshold'); ax.set_ylabel('False Negatives')
ax.set_title('False Negatives vs Threshold', fontweight='bold')
ax.legend(framealpha=0)

ax = axes[1]
ax.plot(thresholds, f1_list,        color=PALETTE[0], linewidth=2.2, label='F1 Score')
ax.plot(thresholds, recall_list,    color=PALETTE[1], linewidth=2.2, label='Recall',    linestyle='--')
ax.plot(thresholds, precision_list, color=PALETTE[3], linewidth=2.2, label='Precision', linestyle=':')
ax.axvline(best_threshold, color='black', linestyle='--', linewidth=1.5, alpha=0.5, label=f'Best ({best_threshold:.2f})')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('F1 / Recall / Precision vs Threshold', fontweight='bold')
ax.legend(framealpha=0); ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('06_threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧠 Step 8 — SHAP Explainability
> SHAP reveals **why** the model predicts churn for each customer — not just which features matter globally.

In [ ]:
explainer  = shap.TreeExplainer(results['XGBoost']['model'])
shap_sample = X_test.iloc[:500]
shap_values = explainer.shap_values(shap_sample)

mean_shap   = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns)
top_features = mean_shap.nlargest(15)

def clean_name(name):
    name = name.replace('Contract_','Contract: ').replace('PaymentMethod_','Payment: ')
    name = name.replace('InternetService_','Internet: ').replace('_',' ').title()
    return name

print("🔑 Top 3 Churn Drivers (SHAP):")
for i, (feat, val) in enumerate(mean_shap.nlargest(3).items(), 1):
    print(f"   {i}. {clean_name(feat):<40} SHAP = {val:.4f}")

In [ ]:
# ── SHAP Bar Chart ──
top_features_clean = top_features.copy()
top_features_clean.index = [clean_name(n) for n in top_features_clean.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('SHAP Feature Importance — XGBoost Churn Model', fontweight='bold', fontsize=14)

ax = axes[0]
colors_shap = [PALETTE[1] if i < 3 else PALETTE[0] for i in range(len(top_features_clean))]
ax.barh(top_features_clean.index[::-1], top_features_clean.values[::-1],
        color=colors_shap[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 15 Churn Drivers\n(pink = top 3)', fontweight='bold')

# SHAP Beeswarm
ax2 = axes[1]
top10 = mean_shap.nlargest(10)
for idx_i, feat in enumerate(top10.index):
    feat_shap = shap_values[:, list(X_test.columns).index(feat)]
    jitter    = np.random.normal(0, 0.07, len(feat_shap))
    ax2.scatter(feat_shap, [idx_i]*len(feat_shap)+jitter,
                c=shap_sample[feat].values, cmap='RdYlGn',
                alpha=0.35, s=8, linewidths=0)

ax2.set_yticks(range(10))
ax2.set_yticklabels([clean_name(n) for n in top10.index], fontsize=9)
ax2.axvline(0, color='gray', linewidth=1, linestyle='--', alpha=0.7)
ax2.set_xlabel('SHAP Value')
ax2.set_title('SHAP Distribution — Top 10 Features\n(green=high value, red=low value)', fontweight='bold')

plt.tight_layout()
plt.savefig('07_shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 📋 Step 9 — Cohort Retention Dashboard

In [ ]:
df_dash = df.copy()
df_dash['Churn_bin'] = (df_dash['Churn']=='Yes').astype(int)
df_dash['TenureBand'] = pd.cut(df_dash['tenure'],
    bins=[0,6,12,24,36,48,72],
    labels=['0-6m','6-12m','12-24m','24-36m','36-48m','48-72m'])

cohort = df_dash.groupby('TenureBand').agg(
    Customers=('customerID','count'),
    ChurnRate=('Churn_bin', lambda x: x.mean()*100),
    AvgMonthly=('MonthlyCharges','mean'),
    TotalRevenue=('TotalCharges','sum')
).reset_index()

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Customer Churn — Cohort Retention Dashboard', fontsize=16, fontweight='bold', y=0.98)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.38)

# Cohort churn rate
ax1 = fig.add_subplot(gs[0, :2])
bar_colors = [PALETTE[1] if r > 25 else PALETTE[0] for r in cohort['ChurnRate']]
bars = ax1.bar(cohort['TenureBand'].astype(str), cohort['ChurnRate'],
               color=bar_colors, edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, cohort['ChurnRate']):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax1.set_xlabel('Tenure Band'); ax1.set_ylabel('Churn Rate (%)')
ax1.set_title('Churn Rate by Tenure Cohort  (red = high risk)', fontweight='bold')
ax1.set_ylim(0, cohort['ChurnRate'].max()*1.3)
ax1.axhline(df_dash['Churn_bin'].mean()*100, color='gray', linestyle='--',
            label=f"Avg {df_dash['Churn_bin'].mean()*100:.1f}%")
ax1.legend(framealpha=0)

# Customer count
ax2 = fig.add_subplot(gs[0, 2])
ax2.barh(cohort['TenureBand'].astype(str), cohort['Customers'],
         color=PALETTE[3], edgecolor='white', alpha=0.85)
for i, val in enumerate(cohort['Customers']):
    ax2.text(val+5, i, str(val), va='center', fontsize=10)
ax2.set_xlabel('# Customers'); ax2.set_title('Customers per Cohort', fontweight='bold')

# Charges by contract
ax3 = fig.add_subplot(gs[1, 0])
cc2 = df_dash.groupby(['Contract','Churn'])['MonthlyCharges'].mean().unstack()
cc2.columns = ['Retained','Churned']
cc2.plot(kind='bar', ax=ax3, color=[PALETTE[0],PALETTE[1]], edgecolor='white', width=0.6, rot=15)
ax3.set_ylabel('Avg Monthly Charges ($)')
ax3.set_title('Avg Charges: Retained vs Churned\nby Contract', fontweight='bold')
ax3.legend(framealpha=0)

# Senior vs Non-Senior
ax4 = fig.add_subplot(gs[1, 1])
sc = df_dash.groupby('SeniorCitizen')['Churn_bin'].mean()*100
ax4.bar(['Non-Senior','Senior'], sc.values, color=[PALETTE[0],PALETTE[1]],
        edgecolor='white', width=0.4)
for i, val in enumerate(sc.values):
    ax4.text(i, val+0.3, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
ax4.set_ylabel('Churn Rate (%)'); ax4.set_title('Senior vs Non-Senior Churn', fontweight='bold')
ax4.set_ylim(0, sc.max()*1.35)

# Revenue at risk
ax5 = fig.add_subplot(gs[1, 2])
rev = df_dash[df_dash['Churn']=='Yes'].groupby('TenureBand')['MonthlyCharges'].sum()
ax5.fill_between(range(len(rev)), rev.values, alpha=0.3, color=PALETTE[1])
ax5.plot(range(len(rev)), rev.values, color=PALETTE[1], linewidth=2.5, marker='o', markersize=7)
ax5.set_xticks(range(len(rev)))
ax5.set_xticklabels(rev.index.astype(str), fontsize=9)
ax5.set_ylabel('Monthly Revenue at Risk ($)')
ax5.set_title('Revenue at Risk by Cohort', fontweight='bold')
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.savefig('08_cohort_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Cohort dashboard saved!")

## ✅ Step 10 — Final Results Summary

In [ ]:
print("=" * 60)
print("  FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"\n  {'Model':<24} {'AUC':>6} {'F1':>6} {'Recall':>8} {'Prec.':>7}")
print(f"  {'-'*53}")
for name, res in results.items():
    print(f"  {name:<24} {res['roc_auc']:>6.3f} {res['f1']:>6.3f} {res['recall']:>8.3f} {res['precision']:>7.3f}")

print(f"\n  Best Model      : XGBoost")
print(f"  Best Threshold  : {best_threshold:.2f}  (tuned from 0.50)")
print(f"  FN Reduction    : {fn_reduction:.1f}%  ({fn_default} → {fn_tuned})")
print(f"\n  Top 3 Churn Drivers (SHAP):")
for i, (feat, val) in enumerate(mean_shap.nlargest(3).items(), 1):
    print(f"    {i}. {clean_name(feat)}")
print(f"\n  Output files saved: 01 to 08 .png charts in current folder")
print("=" * 60)

---
## 📁 Output Files Generated

| File | Description |
|---|---|
| `01_eda_overview.png` | 6-panel EDA — churn rate, tenure, contract, charges |
| `02_correlation_heatmap.png` | Feature correlation matrix |
| `03_model_comparison.png` | All 4 models on 5 metrics |
| `04_roc_curves.png` | ROC curves with AUC scores |
| `05_confusion_matrices.png` | Confusion matrices — all 4 models |
| `06_threshold_tuning.png` | FN reduction via threshold tuning |
| `07_shap_analysis.png` | SHAP bar + beeswarm — top churn drivers |
| `08_cohort_dashboard.png` | Full retention cohort dashboard |

---
*Built by N.V. Mani Deepika — Data Science Portfolio Project*
